In [1]:
#!rm -rf /usr/local/lib/python3.12/dist-packages/pyspark*
#!rm -rf /usr/local/lib/python3.12/dist-packages/py4j*
#!rm -rf /usr/local/lib/python3.12/dist-packages/~yspark*
#!pip install -q pyspark==3.5.1
#!pip install -q synapseml==1.1.3

In [1]:
!pip uninstall -y dataproc-spark-connect google-spark-connect 2>/dev/null
!pip install -q pyspark==3.5.1
!pip install -q synapseml==1.1.3

## Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
from pyspark.storagelevel import StorageLevel
from pyspark.ml.feature import VectorAssembler
from synapse.ml.isolationforest import IsolationForest

spark = (
    SparkSession.builder
    .appName("RAN-LTE-Preprocessing")
    .master("local[2]")
    .config("spark.driver.memory", "10g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:1.1.3")
    .getOrCreate()
)
print("Spark version:", spark.version)

Spark version: 3.5.1


In [3]:
print(spark.sparkContext.getConf().get('spark.jars.packages'))

com.microsoft.azure:synapseml_2.12:1.1.3


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Read the data**

In [5]:
data_root = Path("/content/drive/MyDrive/Dataset/Dataset_01/Dataset_01/Baseband_02/")

csv_files = list(data_root.rglob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file)

Number of CSV files: 1
/content/drive/MyDrive/Dataset/Dataset_01/Dataset_01/Baseband_02/Dataset_01_LTE_2100.csv


In [6]:
for file in csv_files:
    print("\n" + "=" * 80)
    print(file.name)

    temp_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(str(file))
    )
# option("header", True) first row is a header
# option("inferSchema", False) don't infer the data type
    print("Number of columns:", len(temp_df.columns))
    print("Columns:")
    print(temp_df.columns)


Dataset_01_LTE_2100.csv
Number of columns: 19
Columns:
['Base station', 'Sector', 'Timestamp', 'Radio unit energy consumption', 'Baseband energy consumption', '4G max active users DL', '4G max active users UL', '4G data volume DL', '4G data volume UL', '4G max RRC users', '4G RB utilization', '4G CQI rank 1', '4G CQI rank 2', '4G CQI rank 3', '4G CQI rank 4', '4G RRC users', '4G active users UL', '4G active users DL', '4G MIMO rank DL']


**Add technology and Frequency**

In [7]:
%%time

lte_dfs = []

for file in csv_files:

    # Read LTE files only
    if "LTE" not in file.name:
        continue

    print("=" * 80)
    print("Reading:", file.name)

    # Read file
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(file))
    )

    # Extract the frequency from file name
    frequency = file.stem.split("_")[-1]

    #Add Technology & Frequency
    df = (
        df
        .withColumn("Technology", F.lit("4G"))
        .withColumn("Frequency", F.lit(frequency))
    )

    print("Technology:", "4G")
    print("Frequency:", frequency)

    # append
    lte_dfs.append(df)

    # first 3 rows
    df.select(
        "Base station",
        "Sector",
        "Timestamp",
        "Technology",
        "Frequency"
    ).show(3, truncate=False)

Reading: Dataset_01_LTE_2100.csv
Technology: 4G
Frequency: 2100
+------------+------+-------------------+----------+---------+
|Base station|Sector|Timestamp          |Technology|Frequency|
+------------+------+-------------------+----------+---------+
|Site 36     |1     |2023-10-08 06:00:00|4G        |2100     |
|Site 36     |2     |2023-10-08 06:00:00|4G        |2100     |
|Site 36     |3     |2023-10-08 06:00:00|4G        |2100     |
+------------+------+-------------------+----------+---------+
only showing top 3 rows

CPU times: user 9.75 ms, sys: 854 µs, total: 10.6 ms
Wall time: 4.2 s


**Collect all LTE file in one dataframe**

In [8]:
%%time

# first DF
lte_df = lte_dfs[0]

# start from secound one
for df in lte_dfs[1:]:
    lte_df = lte_df.unionByName(
        df,
        allowMissingColumns=True
    )

print("LTE columns:", len(lte_df.columns))
print("LTE rows:", lte_df.count())

lte_df.select(
    "Base station",
    "Sector",
    "Timestamp",
    "Technology",
    "Frequency"
).show(10, truncate=False)

LTE columns: 21
LTE rows: 151026
+------------+------+-------------------+----------+---------+
|Base station|Sector|Timestamp          |Technology|Frequency|
+------------+------+-------------------+----------+---------+
|Site 36     |1     |2023-10-08 06:00:00|4G        |2100     |
|Site 36     |2     |2023-10-08 06:00:00|4G        |2100     |
|Site 36     |3     |2023-10-08 06:00:00|4G        |2100     |
|Site 36     |1     |2023-10-23 14:30:00|4G        |2100     |
|Site 36     |2     |2023-10-23 14:30:00|4G        |2100     |
|Site 36     |3     |2023-10-23 14:30:00|4G        |2100     |
|Site 36     |1     |2023-10-20 09:30:00|4G        |2100     |
|Site 36     |2     |2023-10-20 09:30:00|4G        |2100     |
|Site 36     |3     |2023-10-20 09:30:00|4G        |2100     |
|Site 36     |1     |2023-10-20 12:15:00|4G        |2100     |
+------------+------+-------------------+----------+---------+
only showing top 10 rows

CPU times: user 6.87 ms, sys: 1.34 ms, total: 8.21 ms
Wall 

**Persist**

In [9]:
lte_df = lte_df.persist(StorageLevel.MEMORY_AND_DISK)

In [10]:
%%time

lte_df.count()

CPU times: user 2.65 ms, sys: 96 µs, total: 2.74 ms
Wall time: 4.88 s


151026

# Nulls & Duplicates

In [11]:
%%time

null_counts = lte_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in lte_df.columns
]).collect()[0]

for column in lte_df.columns:
    if null_counts[column] > 0:
        print(f"{column}: {null_counts[column]}")

Radio unit energy consumption: 3574
Baseband energy consumption: 3556
4G max active users DL: 3556
4G max active users UL: 3556
4G data volume DL: 3556
4G data volume UL: 3556
4G max RRC users: 3556
4G RB utilization: 16597
4G CQI rank 1: 3556
4G CQI rank 2: 3556
4G CQI rank 3: 3556
4G CQI rank 4: 3556
4G RRC users: 3556
4G active users UL: 16968
4G active users DL: 17236
4G MIMO rank DL: 3556
CPU times: user 31.6 ms, sys: 8.21 ms, total: 39.8 ms
Wall time: 2.96 s


In [12]:
%%time

# Number of duplicate rows in  "Base station", "Sector", "Timestamp", "Frequency"
lte_df.groupBy(
    "Base station", "Sector", "Timestamp", "Frequency"
).count().filter(F.col("count") > 1).count()

CPU times: user 6.07 ms, sys: 0 ns, total: 6.07 ms
Wall time: 2.76 s


0

In [13]:
%%time

lte_df = lte_df.dropDuplicates([
    "Base station", "Sector", "Timestamp", "Frequency"
])

lte_df.count()

CPU times: user 2.68 ms, sys: 0 ns, total: 2.68 ms
Wall time: 1.83 s


151026

In [14]:
%%time
lte_df.count()

CPU times: user 4.44 ms, sys: 360 µs, total: 4.8 ms
Wall time: 1.12 s


151026

## Median Imputation

In [15]:
numeric_cols = [
    field.name
    for field in lte_df.schema.fields
    if field.dataType.simpleString() in ["double", "int", "bigint", "float"]
]

print(numeric_cols)

['Sector', 'Radio unit energy consumption', 'Baseband energy consumption', '4G max active users DL', '4G max active users UL', '4G data volume DL', '4G data volume UL', '4G max RRC users', '4G RB utilization', '4G CQI rank 1', '4G CQI rank 2', '4G CQI rank 3', '4G CQI rank 4', '4G RRC users', '4G active users UL', '4G active users DL', '4G MIMO rank DL']


In [16]:
%%time

median_values = lte_df.approxQuantile(
    numeric_cols,
    [0.5],
    0.01
)

median_dict = {
    feature: value[0]
    for feature, value in zip(numeric_cols, median_values)
}

CPU times: user 77.4 ms, sys: 21.5 ms, total: 98.9 ms
Wall time: 7.29 s


In [17]:
# check Nulls
lte_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in numeric_cols
    ]
).show()

+------+-----------------------------+---------------------------+----------------------+----------------------+-----------------+-----------------+----------------+-----------------+-------------+-------------+-------------+-------------+------------+------------------+------------------+---------------+
|Sector|Radio unit energy consumption|Baseband energy consumption|4G max active users DL|4G max active users UL|4G data volume DL|4G data volume UL|4G max RRC users|4G RB utilization|4G CQI rank 1|4G CQI rank 2|4G CQI rank 3|4G CQI rank 4|4G RRC users|4G active users UL|4G active users DL|4G MIMO rank DL|
+------+-----------------------------+---------------------------+----------------------+----------------------+-----------------+-----------------+----------------+-----------------+-------------+-------------+-------------+-------------+------------+------------------+------------------+---------------+
|     0|                         3574|                       3556|             

In [18]:
lte_df = lte_df.fillna(median_dict)

In [19]:
lte_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in numeric_cols
    ]
).show()

+------+-----------------------------+---------------------------+----------------------+----------------------+-----------------+-----------------+----------------+-----------------+-------------+-------------+-------------+-------------+------------+------------------+------------------+---------------+
|Sector|Radio unit energy consumption|Baseband energy consumption|4G max active users DL|4G max active users UL|4G data volume DL|4G data volume UL|4G max RRC users|4G RB utilization|4G CQI rank 1|4G CQI rank 2|4G CQI rank 3|4G CQI rank 4|4G RRC users|4G active users UL|4G active users DL|4G MIMO rank DL|
+------+-----------------------------+---------------------------+----------------------+----------------------+-----------------+-----------------+----------------+-----------------+-------------+-------------+-------------+-------------+------------+------------------+------------------+---------------+
|     0|                            0|                          0|             

## Feature Engineering

In [20]:
# Time features
lte_df = (
    lte_df
    .withColumn("Hour", F.hour("Timestamp"))
    .withColumn("DayOfWeek", F.dayofweek("Timestamp"))
    .withColumn(
        "TimePeriod",
        F.when(F.col("Hour") < 6, "Night")
         .when(F.col("Hour") < 12, "Morning")
         .when(F.col("Hour") < 18, "Afternoon")
         .otherwise("Evening")
    )
    .withColumn("Hour_sin", F.sin(2 * F.pi() * F.col("Hour") / 24))
    .withColumn("Hour_cos", F.cos(2 * F.pi() * F.col("Hour") / 24))
)

In [21]:
# Percentage of volume and energy
lte_df = (
    lte_df
    .withColumn(
        "Radio Energy %",
        F.when(
            (F.col("Radio unit energy consumption") + F.col("Baseband energy consumption")) > 0,
            F.col("Radio unit energy consumption") /
            (F.col("Radio unit energy consumption") + F.col("Baseband energy consumption")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "Baseband Energy %",
        F.when(
            (F.col("Radio unit energy consumption") + F.col("Baseband energy consumption")) > 0,
            F.col("Baseband energy consumption") /
            (F.col("Radio unit energy consumption") + F.col("Baseband energy consumption")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "DL Traffic %",
        F.when(
            (F.col("4G data volume DL") + F.col("4G data volume UL")) > 0,
            F.col("4G data volume DL") /
            (F.col("4G data volume DL") + F.col("4G data volume UL")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "UL Traffic %",
        F.when(
            (F.col("4G data volume DL") + F.col("4G data volume UL")) > 0,
            F.col("4G data volume UL") /
            (F.col("4G data volume DL") + F.col("4G data volume UL")) * 100
        ).otherwise(0)
    )
)

In [22]:
lte_df.select(
    "Timestamp",
    "Hour",
    "DayOfWeek",
    "TimePeriod",
    "Hour_sin",
    "Hour_cos",
    "Radio Energy %",
    "Baseband Energy %",
    "DL Traffic %",
    "UL Traffic %"
).show(10, truncate=False)

+-------------------+----+---------+----------+----------------------+---------------------+------------------+-----------------+-----------------+------------------+
|Timestamp          |Hour|DayOfWeek|TimePeriod|Hour_sin              |Hour_cos             |Radio Energy %    |Baseband Energy %|DL Traffic %     |UL Traffic %      |
+-------------------+----+---------+----------+----------------------+---------------------+------------------+-----------------+-----------------+------------------+
|2023-10-09 07:45:00|7   |2        |Morning   |0.9659258262890683    |-0.25881904510252063 |37.404580152671755|62.59541984732825|87.52736041550733|12.472639584492672|
|2023-10-20 12:45:00|12  |6        |Afternoon |1.2246467991473532E-16|-1.0                 |52.48618784530387 |47.51381215469613|94.9395562031817 |5.0604437968183005|
|2023-10-19 06:15:00|6   |5        |Morning   |1.0                   |6.123233995736766E-17|31.666666666666664|68.33333333333333|95.61874249098918|4.381257509010813 

In [23]:
#we found infinite value in 4G active users DL
lte_df.select(
    F.sum(
        F.when(
            F.col("4G active users DL") == float("inf"),
            1
        ).otherwise(0)
    ).alias("positive_inf"),

    F.sum(
        F.when(
            F.col("4G active users DL") == float("-inf"),
            1
        ).otherwise(0)
    ).alias("negative_inf")
).show()

+------------+------------+
|positive_inf|negative_inf|
+------------+------------+
|           0|           0|
+------------+------------+



In [24]:
# Replace inf with median
lte_df = lte_df.withColumn(
    "4G active users DL",
    F.when(
        F.col("4G active users DL") == float("inf"),
        F.lit(median_dict["4G active users DL"])
    ).otherwise(F.col("4G active users DL"))
)

**Persist**

In [25]:
lte_df.unpersist()
lte_df = lte_df.persist(StorageLevel.MEMORY_AND_DISK)

In [26]:
%%time
lte_df.count()

CPU times: user 7.63 ms, sys: 836 µs, total: 8.47 ms
Wall time: 24.7 s


151026

## Behavioral Baseline (context-aware)

In [27]:
baseline_df = (
    lte_df
    .groupBy("Base station", "Sector", "Frequency", "DayOfWeek", "Hour")
    .agg(
        F.avg("4G data volume DL").alias("DL_mean"),
        F.stddev("4G data volume DL").alias("DL_std"),

        F.avg("4G data volume UL").alias("UL_mean"),
        F.stddev("4G data volume UL").alias("UL_std"),

        F.avg("4G RB utilization").alias("RB_mean"),
        F.stddev("4G RB utilization").alias("RB_std"),

        F.avg("4G RRC users").alias("RRC_mean"),
        F.stddev("4G RRC users").alias("RRC_std"),

        F.avg("4G active users DL").alias("ActiveDL_mean"),
        F.stddev("4G active users DL").alias("ActiveDL_std"),

        F.avg("4G active users UL").alias("ActiveUL_mean"),
        F.stddev("4G active users UL").alias("ActiveUL_std"),

        F.avg("4G MIMO rank DL").alias("MIMO_mean"),
        F.stddev("4G MIMO rank DL").alias("MIMO_std"),

        F.avg("Radio unit energy consumption").alias("RadioEnergy_mean"),
        F.stddev("Radio unit energy consumption").alias("RadioEnergy_std"),

        F.avg("Baseband energy consumption").alias("BasebandEnergy_mean"),
        F.stddev("Baseband energy consumption").alias("BasebandEnergy_std"),
    )
)

baseline_keys = ["Base station", "Sector", "Frequency", "DayOfWeek", "Hour"]

lte_baseline = lte_df.join(baseline_df, on=baseline_keys, how="left")

In [28]:
# Z-Score
lte_anomaly = (
    lte_baseline
    .withColumn("DL_zscore", F.when(F.col("DL_std") > 0,
        (F.col("4G data volume DL") - F.col("DL_mean")) / F.col("DL_std")).otherwise(0.0))
    .withColumn("UL_zscore", F.when(F.col("UL_std") > 0,
        (F.col("4G data volume UL") - F.col("UL_mean")) / F.col("UL_std")).otherwise(0.0))
    .withColumn("RB_zscore", F.when(F.col("RB_std") > 0,
        (F.col("4G RB utilization") - F.col("RB_mean")) / F.col("RB_std")).otherwise(0.0))
    .withColumn("RRC_zscore", F.when(F.col("RRC_std") > 0,
        (F.col("4G RRC users") - F.col("RRC_mean")) / F.col("RRC_std")).otherwise(0.0))
    .withColumn("ActiveDL_zscore", F.when(F.col("ActiveDL_std") > 0,
        (F.col("4G active users DL") - F.col("ActiveDL_mean")) / F.col("ActiveDL_std")).otherwise(0.0))
    .withColumn("ActiveUL_zscore", F.when(F.col("ActiveUL_std") > 0,
        (F.col("4G active users UL") - F.col("ActiveUL_mean")) / F.col("ActiveUL_std")).otherwise(0.0))
    .withColumn("MIMO_zscore", F.when(F.col("MIMO_std") > 0,
        (F.col("4G MIMO rank DL") - F.col("MIMO_mean")) / F.col("MIMO_std")).otherwise(0.0))
    .withColumn("RadioEnergy_zscore", F.when(F.col("RadioEnergy_std") > 0,
        (F.col("Radio unit energy consumption") - F.col("RadioEnergy_mean")) / F.col("RadioEnergy_std")).otherwise(0.0))
    .withColumn("BasebandEnergy_zscore", F.when(F.col("BasebandEnergy_std") > 0,
        (F.col("Baseband energy consumption") - F.col("BasebandEnergy_mean")) / F.col("BasebandEnergy_std")).otherwise(0.0))
)

**Feature Vector**

In [29]:
feature_cols = [
    # Energy
    "Radio unit energy consumption",
    "Baseband energy consumption",
    "Radio Energy %",
    "Baseband Energy %",

    # Users
    "4G max active users DL",
    "4G max active users UL",
    "4G max RRC users",
    "4G RRC users",
    "4G active users DL",
    "4G active users UL",

    # Traffic
    "4G data volume DL",
    "4G data volume UL",
    "DL Traffic %",
    "UL Traffic %",

    # Radio quality
    "4G RB utilization",
    "4G CQI rank 1",
    "4G CQI rank 2",
    "4G CQI rank 3",
    "4G CQI rank 4",
    "4G MIMO rank DL",

    # Time (cyclical)
    "Hour_sin",
    "Hour_cos",

    # Context-aware anomaly signal
    "DL_zscore",
    "UL_zscore",
    "RB_zscore",
    "RRC_zscore",
    "ActiveDL_zscore",
    "ActiveUL_zscore",
    "MIMO_zscore",
    "RadioEnergy_zscore",
    "BasebandEnergy_zscore",
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip",   # يرمي أي صف لسه فيه null/NaN بدل ما يفشل الفيت
)

lte_features = assembler.transform(lte_anomaly)

In [30]:
lte_features = lte_features.persist(StorageLevel.MEMORY_AND_DISK)

In [31]:
lte_features.count()

151026

## Isolation Forest

In [32]:
print(spark.sparkContext.getConf().get('spark.jars.packages'))

com.microsoft.azure:synapseml_2.12:1.1.3


In [33]:
isolation_forest = (
    IsolationForest()
    .setNumEstimators(100)
    .setBootstrap(False)
    .setMaxSamples(256)
    .setMaxFeatures(1.0)
    .setFeaturesCol("features")
    .setPredictionCol("predictedLabel")
    .setScoreCol("outlierScore")
    .setContamination(0.02)
    .setContaminationError(0.02 * 0.01)
    .setRandomSeed(1)
)

In [34]:
print(spark.sparkContext.getConf().get('spark.jars.packages'))

com.microsoft.azure:synapseml_2.12:1.1.3


In [35]:
%%time
isolation_forest_model = isolation_forest.fit(lte_features)

CPU times: user 56.3 ms, sys: 8.86 ms, total: 65.1 ms
Wall time: 1min 13s


In [36]:
%%time
predictions = isolation_forest_model.transform(lte_features)
predictions = predictions.persist(StorageLevel.MEMORY_AND_DISK)
predictions.count()

CPU times: user 19.6 ms, sys: 8.64 ms, total: 28.3 ms
Wall time: 27.1 s


151026

# Isolation Forest Result

In [37]:
predictions.groupBy("predictedLabel").count().show()

+--------------+------+
|predictedLabel| count|
+--------------+------+
|           0.0|147990|
|           1.0|  3036|
+--------------+------+



In [38]:
predictions.filter(
    F.col("predictedLabel") == 1
).orderBy(
    F.col("outlierScore").desc()
).show(20, truncate=False)

+------------+------+---------+---------+----+-------------------+-----------------------------+---------------------------+----------------------+----------------------+-----------------+-----------------+----------------+-----------------+-------------+-------------+-------------+-------------+------------+------------------+------------------+---------------+----------+----------+----------------------+-----------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+--------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+--------------------+------